## 02 - Data Cleaning

In [28]:
import pandas as pd
import numpy as np

In [10]:
df=pd.read_csv("../data/processed/online_retail_II_processed.csv")

In [7]:
df.columns

Index(['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'Price', 'Customer ID', 'Country'],
      dtype='str')

In [12]:
df.dtypes

Invoice            str
StockCode          str
Description        str
Quantity         int64
InvoiceDate        str
Price          float64
Customer ID    float64
Country            str
dtype: object

### Column Renaming

In [13]:
df=df.rename(columns={
    'Invoice':'invoice_number', 
    'StockCode':'stock_code',
    'Description':'description',
    'Quantity':'quantity',
    'InvoiceDate':'invoice_date',
    'Price':'unit_price',
    'Customer ID':'customer_id',
    'Country':'country'
})

### Normalizing Datatypes

In [14]:
df=df.astype({
    'invoice_number':'string', 
    'stock_code':'string',
    'description':'string',
    'quantity':'Int64',
    'invoice_date':'datetime64[ns]',
    'unit_price':'float64',
    'customer_id':"Int64",
    'country':'string'
})

### Removing exact duplicates

In [17]:
df.duplicated().sum()

np.int64(34335)

In [18]:
df = df.drop_duplicates(keep="first")

In [21]:
df.shape

(1033036, 8)

### Normalize cancelled positive quantities

In [29]:
df.loc[(df["invoice_number"].str.startswith("C")) & (df["quantity"]>0), "quantity"] *=-1

### Normalize the Stockcode

In [25]:
df["stock_code"].str.upper()

0           85048
1          79323P
2          79323W
3           22041
4           21232
            ...  
1067366     22899
1067367     23254
1067368     23255
1067369     22138
1067370      POST
Name: stock_code, Length: 1033036, dtype: string

### Creating and Classify transaction status

In [30]:
conditions=[
    df["invoice_number"].str.startswith("C",na=False),
    df["quantity"]<0
]
choices=[
    "Cancelled",
    "Return/Adjusted"
]

df["transaction_status"]= np.select(
    conditions,
    choices,
    default="Completed"
)

### Excluding invalid/test records

In [31]:
remove_codes=[
    "TEST001",
    "TEST002",
    "GIFT",
    "DCGSLGIRL",
    "DCGSLBOY"
]
df=df[~df["stock_code"].isin(remove_codes)]

### Extracting Date, year, month, and Time from invoice date

In [ ]:
df["transaction_date"]=df["invoice_date"].dt.date

In [ ]:
df["transaction_month"]=df["invoice_date"].dt.month

In [ ]:
df["transaction_year"]=df["invoice_date"].dt.year

In [36]:
df["transaction_time"]=df["invoice_date"].dt.time

### Calculating revenue fields

In [37]:
df["gross_revenue"]=np.where(
    df["quantity"]>0,
    df['quantity']*df['unit_price'],
    0
)

In [38]:
df['return_value']=np.where(
    df['quantity']<0,
    abs(df['quantity']*df['unit_price']),
    0
)

In [39]:
df['net_revenue']=df['gross_revenue']-df['return_value']

### Saving Dataset

In [40]:
df.to_csv("../data/cleaned/online_retail_II_cleaned.csv", index=False)